# Notebook 05: Preprocesamiento de variables meteorológicas

Preparamos las variables meteorológicas horarias de la estación SUMU (Aeropuerto de Carrasco) que se usarán como exógenas en el notebook 06. Los datos meteorológicos son los mismos para Curva de Maroñas y Museo Romántico.

## 1. Configuración y conservación de los datos originales
Se descarga una copia sin modificar de `meteorologia_original.csv` en `data/raw/` en modo solo lectura, y se trabaja sobre una copia en memoria. Los timestamps se pasan a la zona horaria de Montevideo, igual que la serie de PM2.5.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from urllib.request import urlretrieve

SEED = 42
rng = np.random.default_rng(SEED)

URL_METEO = "https://raw.githubusercontent.com/trajano-carballo/paa-pronostico-pm25-montevideo/main/data/raw/meteorologia_original.csv"
DIR_RAW, DIR_PROC = Path("data/raw"), Path("data/processed")
DIR_RAW.mkdir(parents=True, exist_ok=True); DIR_PROC.mkdir(parents=True, exist_ok=True)

copia = DIR_RAW / "meteorologia_original.csv"
if not copia.exists():
    urlretrieve(URL_METEO, copia)
    copia.chmod(0o444)

met_raw = pd.read_csv(copia)
met = met_raw.copy()
met["hora"] = pd.to_datetime(met["timestamp"], utc=True).dt.tz_convert("America/Montevideo")
met = met.drop(columns="timestamp").set_index("hora").sort_index()

VARS = ["temperatura_c", "humedad_relativa_pct", "direccion_viento_grados",
        "velocidad_viento_nudos", "visibilidad_millas"]
met.head()

,temperatura_c,humedad_relativa_pct,direccion_viento_grados,velocidad_viento_nudos,visibilidad_millas,indicador_valor_faltante
hora,,,,,,
2024-01-01 00:00:00-03:00,19.0,68.15,40.0,4.0,6.21,f
2024-01-01 01:00:00-03:00,18.0,72.56,50.0,7.0,6.21,f
2024-01-01 02:00:00-03:00,18.0,72.56,30.0,7.0,6.21,f
2024-01-01 03:00:00-03:00,17.0,77.29,30.0,5.0,6.21,f
2024-01-01 04:00:00-03:00,16.0,87.92,10.0,3.0,6.21,f


## 2. Diagnóstico
### 2.1 Índice temporal y horas sin registro
Se verifica que no haya horas duplicadas y se detectan las horas que directamente no existen en el archivo (faltantes implícitos), agrupadas en huecos consecutivos.

In [2]:
grilla = pd.date_range(met.index.min(), met.index.max(), freq="h", name="hora")
faltan = grilla.difference(met.index)
print(f"Duplicados: {met.index.duplicated().sum()} | horas esperadas: {len(grilla)} | sin registro: {len(faltan)}")

bloque = (pd.Series(faltan).diff() != pd.Timedelta("1h")).cumsum()
huecos_impl = pd.Series(faltan).groupby(bloque.values).agg(inicio="min", fin="max", horas="size")
print(huecos_impl["horas"].value_counts().sort_index().rename("huecos implícitos por largo (h)"))
huecos_impl.sort_values("horas", ascending=False).head()

Duplicados: 0 | horas esperadas: 21024 | sin registro: 47
horas
1    28
2     3
3     1
4     1
6     1
Name: huecos implícitos por largo (h), dtype: int64


,inicio,fin,horas
1,2024-01-07 23:00:00-03:00,2024-01-08 04:00:00-03:00,6
26,2025-06-16 21:00:00-03:00,2025-06-17 00:00:00-03:00,4
28,2025-08-04 22:00:00-03:00,2025-08-05 00:00:00-03:00,3
27,2025-06-22 03:00:00-03:00,2025-06-22 04:00:00-03:00,2
15,2024-06-24 00:00:00-03:00,2024-06-24 01:00:00-03:00,2


### 2.2 Filas marcadas y faltantes explícitos
`indicador_valor_faltante = "t"` marca observaciones no utilizables. Además, algunas variables pueden venir vacías de forma independiente.

In [3]:
marcadas = met["indicador_valor_faltante"].eq("t")
print(f"Filas marcadas como no utilizables: {marcadas.sum()}")
met[VARS].isna().sum().rename("NaN explícitos")

Filas marcadas como no utilizables: 10


,NaN explícitos
temperatura_c,1
humedad_relativa_pct,2
direccion_viento_grados,8
velocidad_viento_nudos,0
visibilidad_millas,0


### 2.3 Rangos de valores posibles y casos particulares del viento

Los datos provienen de los reportes METAR de la estación SUMU (Aeropuerto Internacional de Carrasco). Un METAR es el reporte meteorológico de rutina que emiten los aeropuertos, con un formato codificado estándar de la aviación: informa, entre otras cosas, la dirección y velocidad del viento (promedio de los 10 minutos previos a la observación, con dirección redondeada a decenas de grados), la visibilidad, la temperatura y el punto de rocío (IMGW, s.f.). Ese formato explica varias particularidades de la serie:

- **Viento en calma:** se codifica como `00000KT`, es decir, dirección 0 y velocidad 0. Por eso una dirección de 0° solo indica viento del norte si la velocidad es mayor que cero.
- **Viento variable (VRB):** cuando la dirección es inestable se reporta `VRB` en lugar de un ángulo; en nuestra base llega como dirección vacía.
- **Visibilidad:** el código `9999` significa "10 km o más", por lo que 10 km (6.21 millas) es un tope de registro y no una medición exacta. La variable queda censurada en ese valor.

Para detectar errores de registro se definen, de antemano y sin aprenderlos de los datos, los rangos de valores posibles para cada variable:

| Variable | Rango | Fundamento |
|---|---|---|
| Humedad relativa | 0 a 100 % | Definición de la variable |
| Dirección del viento | 0° a 360° | Definición de la variable |
| Visibilidad | 0 a 10 km | Tope de codificación METAR (`9999`) |
| Temperatura | −10 °C a 45 °C | Extremos registrados en Uruguay: −8.0 °C (Mercedes, 1993; INUMET, s.f.) y 42.5 °C (Salto, 2022; Montevideo Portal, 2025), con margen |
| Velocidad del viento (media) | 0 a 60 nudos (≈111 km/h) | En el ciclón de agosto de 2026, las ráfagas máximas en la zona de Carrasco rondaron 85-87 km/h (≈47 nudos; Montevideo Portal, 2026). La velocidad media de METAR es menor que las ráfagas, por lo que 60 nudos es un límite holgado |

Un valor fuera de estos rangos no puede ser una medición real y se trata como error de registro. Dentro de los rangos no se elimina ningún valor por criterios estadísticos.

In [4]:
display(met[VARS].describe().T[["min", "50%", "max"]])

v, d = met["velocidad_viento_nudos"], met["direccion_viento_grados"]
display(pd.Series({
    "calma (velocidad 0)":              (v == 0).sum(),
    "dirección 360°":                   (d == 360).sum(),
    "dirección vacía con viento (VRB)": (d.isna() & (v > 0)).sum(),
    "visibilidad en el tope (6.21 mi)": (met["visibilidad_millas"] >= 6.21).sum(),
}))

RANGOS = {"temperatura_c": (-10, 45), "humedad_relativa_pct": (0, 100),
          "direccion_viento_grados": (0, 360), "velocidad_viento_nudos": (0, 60),
          "visibilidad_millas": (0, 6.21)}
fuera = pd.DataFrame({c: ~met[c].between(lo, hi) & met[c].notna() for c, (lo, hi) in RANGOS.items()})
display(fuera.sum().rename("fuera de rango"))
met.loc[fuera.any(axis=1), VARS]

,min,50%,max
temperatura_c,-3.00,18.00,38.00
humedad_relativa_pct,15.99,83.00,100.00
direccion_viento_grados,0.00,120.00,360.00
velocidad_viento_nudos,0.00,7.00,162.00
visibilidad_millas,0.00,6.21,6.21


,0
calma (velocidad 0),3369
dirección 360°,391
dirección vacía con viento (VRB),8
visibilidad en el tope (6.21 mi),18211


,fuera de rango
temperatura_c,0
humedad_relativa_pct,0
direccion_viento_grados,0
velocidad_viento_nudos,1
visibilidad_millas,0


,temperatura_c,humedad_relativa_pct,direccion_viento_grados,velocidad_viento_nudos,visibilidad_millas
hora,,,,,
2025-12-20 12:00:00-03:00,36.0,26.94,20.0,162.0,6.21


## 3. Limpieza
Las filas marcadas y los valores fuera de rango físico pasan a faltantes. El único caso fuera de rango es un viento de 162 nudos (≈300 km/h) el 20/12/2025 a las 12:00, físicamente imposible en Montevideo y muy por encima del siguiente máximo de la serie (32 nudos), por lo que se trata como error de registro. No se eliminan valores mediante criterios estadísticos. Finalmente, se completa la grilla horaria para que las horas sin registro aparezcan como faltantes.

In [5]:
met.loc[marcadas, VARS] = np.nan
for c in VARS:
    met.loc[fuera[c], c] = np.nan
met = met.drop(columns="indicador_valor_faltante").reindex(grilla)
met[VARS].isna().sum().rename("NaN tras la limpieza")

,NaN tras la limpieza
temperatura_c,57
humedad_relativa_pct,57
direccion_viento_grados,57
velocidad_viento_nudos,58
visibilidad_millas,57


## 4. Transformaciones
- **Unidades:** la velocidad del viento pasa de nudos a m/s y la visibilidad de millas a km.
- **Dirección del viento:** es una variable circular: 359° y 1° son casi la misma dirección, pero como números están muy lejos, y un modelo los trataría como opuestos. Adoptamos el criterio de Blazakis et al. (2025), que tratan la dirección como variable circular representándola con su seno y su coseno, de modo que los ángulos quedan sobre un círculo continuo, sin el salto entre 360° y 0°. Nosotros multiplicamos esas dos componentes por la velocidad del viento, con lo que se obtienen `viento_u` (componente este-oeste) y `viento_v` (componente norte-sur), en m/s, que indican hacia dónde se desplaza el aire y con qué fuerza. Esto resuelve además el caso del viento en calma: su dirección se registra como 0°, que con seno y coseno se confundiría con viento del norte, mientras que con velocidad 0 ambas componentes valen 0. La dirección en grados se descarta.
- **Velocidad:** se conserva como variable propia, porque la intensidad del viento influye por sí sola en la dispersión de los contaminantes.
- **Visibilidad:** se mantiene, teniendo en cuenta que está censurada en 10 km.

In [6]:
met["velocidad_viento_ms"] = met["velocidad_viento_nudos"] * 0.514444
met["visibilidad_km"] = met["visibilidad_millas"] * 1.609344

rad = np.radians(met["direccion_viento_grados"])
met["viento_u"] = -met["velocidad_viento_ms"] * np.sin(rad)   # + hacia el este
met["viento_v"] = -met["velocidad_viento_ms"] * np.cos(rad)   # + hacia el norte
met.loc[met["velocidad_viento_ms"].eq(0), ["viento_u", "viento_v"]] = 0.0   # calma

met = met.rename(columns={"temperatura_c": "temperatura", "humedad_relativa_pct": "humedad"})
EXOG = ["temperatura", "humedad", "velocidad_viento_ms", "viento_u", "viento_v", "visibilidad_km"]
met[EXOG].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
temperatura,20967.0,17.42,6.24,-3.00,13.00,18.00,22.00,38.00
humedad,20967.0,80.65,17.85,15.99,68.38,83.00,100.00,100.00
velocidad_viento_ms,20966.0,3.61,2.48,0.00,2.06,3.60,5.14,16.46
viento_u,20966.0,-0.96,2.89,-11.12,-2.83,-0.79,0.00,13.38
viento_v,20966.0,0.71,3.07,-13.17,-1.32,0.00,2.90,15.43
visibilidad_km,20967.0,9.30,2.08,0.00,9.99,9.99,9.99,9.99


## 5. Partición y tratamiento de faltantes
Se aplica el mismo criterio que a PM2.5: se interpolan linealmente solo los huecos de hasta 3 h, por separado en desarrollo (2024-2025) y test (2026), y los huecos mayores quedan como faltantes. Las componentes `viento_u` y `viento_v` se interpolan como cualquier variable continua; la dirección en grados nunca se interpola. La columna `exog_interpolada` registra qué horas fueron rellenadas.

In [7]:
CORTE = pd.Timestamp("2026-01-01", tz="America/Montevideo")
met["particion"] = np.where(met.index < CORTE, "desarrollo", "test")

def interpolar_huecos_cortos(s, max_horas=3):
    interp = s.interpolate(method="linear", limit_area="inside")
    nan = s.isna()
    bloque = nan.ne(nan.shift()).cumsum()
    largo = nan.groupby(bloque).transform("sum")
    return s.where(~(nan & (largo <= max_horas)), interp)

antes = met[EXOG].isna()
for c in EXOG:
    met[c] = met.groupby("particion")[c].transform(interpolar_huecos_cortos)
interp = antes & met[EXOG].notna()
met["exog_interpolada"] = interp.any(axis=1)

pd.DataFrame({"NaN antes": antes.sum(), "interpolados": interp.sum(),
              "NaN restantes": met[EXOG].isna().sum()})

,NaN antes,interpolados,NaN restantes
temperatura,57,47,10
humedad,57,47,10
velocidad_viento_ms,58,48,10
viento_u,58,48,10
viento_v,58,48,10
visibilidad_km,57,47,10


## 6. Resumen, verificación y guardado
Se verifica que la grilla sea horaria y sin duplicados, que todas las horas de PM2.5 tengan su fila meteorológica, y se guarda el conjunto procesado.

In [8]:
salida = met[EXOG + ["exog_interpolada", "particion"]].reset_index()
assert salida["hora"].is_unique and salida["hora"].diff().dropna().eq(pd.Timedelta("1h")).all()

display(salida.groupby("particion").agg(inicio=("hora", "min"), fin=("hora", "max"),
        horas=("hora", "size"), interpoladas=("exog_interpolada", "sum"),
        horas_con_nan=("temperatura", lambda s: s.isna().sum())))

# Alineación con la serie de PM2.5
URL_PM = "https://raw.githubusercontent.com/trajano-carballo/paa-pronostico-pm25-montevideo/main/data/processed/pm25_horario_preprocesado.csv"
pm = pd.read_csv(URL_PM)
pm["hora"] = pd.to_datetime(pm["hora"], utc=True).dt.tz_convert("America/Montevideo")
for est, g in pm.groupby("estacion"):
    print(f"{est}: horas de PM2.5 sin fila meteorológica = {(~g['hora'].isin(salida['hora'])).sum()}")

salida.to_csv(DIR_PROC / "meteorologia_horaria_preprocesada.csv", index=False)

,inicio,fin,horas,interpoladas,horas_con_nan
particion,,,,,
desarrollo,2024-01-01 00:00:00-03:00,2025-12-31 23:00:00-03:00,17544,43,10
test,2026-01-01 00:00:00-03:00,2026-05-25 23:00:00-03:00,3480,5,0


Curva de Maroñas: horas de PM2.5 sin fila meteorológica = 0
Museo Romántico: horas de PM2.5 sin fila meteorológica = 0


In [9]:
from google.colab import files
files.download("data/processed/meteorologia_horaria_preprocesada.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Referencias

Blazakis, K., Perrakis, S., Karapidakis, E., Mavromatakis, F., & Katsigiannis, Y. (2025). ML-based multi-horizon wind speed and wind direction forecasting for aviation and energy applications in coastal Crete. *Electronics*, *14*(24), Artículo 4856. https://doi.org/10.3390/electronics14244856

Instituto de Meteorología y Gestión del Agua de Polonia [IMGW-PIB]. (s.f.). *Detailed description of METAR*. https://awiacja.imgw.pl/uploads/Detailed_description_METAR_84b34cfe19.pdf

Instituto Uruguayo de Meteorología [INUMET]. (s.f.). *Climatología del invierno (JUN-JUL-AGO) en Uruguay 1981-2010*. https://www.inumet.gub.uy/sites/default/files/2021-06/CLIMATOLOGIA%20DEL%20INVIERNO_LQ.pdf

Montevideo Portal. (2025, 10 de febrero). ¿Se romperá hoy el récord nacional de calor? El litoral tiene "boletos" en esa apuesta. https://www.montevideo.com.uy/Tiempo-libre/-Se-rompera-hoy-el-record-nacional-de-calor-El-litoral-tiene-boletos-en-esa-apuesta-uc914544

Montevideo Portal. (2026, 7 de agosto). Las cuentas del ciclón: ¿hubo o no vientos superiores a los 100 km/h? https://www.montevideo.com.uy/Noticias/Las-cuentas-del-ciclon--hubo-o-no-vientos-superiores-a-los-100-km-h--uc971017